# Speech Emotion Recognition (SER) Ultimate Stacking Ensemble Pipeline

This notebook implements the complete training and evaluation pipeline for our **Ultimate Stacking Ensemble Classifier** (combining **XGBoost**, **LightGBM**, **CatBoost**, **MLP**, and **Random Forest** using an **Extra Trees meta-classifier**).

### Key Improvements:
1. **10-Fold Stratified CV** (up from 3-Fold) for high-quality out-of-fold prediction probabilities.
2. **5 Diverse Base Classifiers** (Gradient Boosting, Bagging, and Neural representations).
3. **Tuned Extra Trees Meta-Classifier**.
4. **Detailed Visualizations** (Confusion matrices, comparison charts).

In [ ]:
%pip install seaborn

In [ ]:
# Multi-Layer Perceptron (MLP) neural network, and a Random Forest classifier.
#
# Peak performance configurations implemented:
# 1. 10-Fold Stratified CV (base models train on 90% of data per fold).
# 2. 5 Base Classifiers (Boosting, Bagging, and Neural representations).
# 3. Optimized Extra Trees Meta-Classifier.
# 4. 6 Side-by-side confusion matrix visualizations.

import json
import os
import warnings
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import torch

warnings.filterwarnings("ignore")

# Global Constants & Hardware Configuration
RANDOM_STATE = 42
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA Available (PyTorch): {CUDA_AVAILABLE}")



## Path Configuration
Set up absolute and relative directories for loading the dataset and parameters.

In [ ]:
# Path Configuration
_base = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() in ["src", "notebooks"] else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))
if not os.path.isfile(ALL_EMOTIONS_CSV):
    fallback_csv = os.path.normpath(os.path.join(_project_root, "all_emotions.csv"))
    if os.path.isfile(fallback_csv):
        ALL_EMOTIONS_CSV = fallback_csv

print(f"Dataset path: {ALL_EMOTIONS_CSV}")
print(f"File exists: {os.path.isfile(ALL_EMOTIONS_CSV)}")



## Data Loading & Imputation
Load all 48 acoustic features and handle missing values.

In [ ]:
# Data Loading & Imputation
df = pd.read_csv(ALL_EMOTIONS_CSV)
target_col = "label" if "label" in df.columns else "Label"

df_cleaned = df.dropna(subset=[target_col]).copy()
df_cleaned = df_cleaned[df_cleaned[target_col].astype(str).str.strip().str.lower() != "nan"]

FEATURE_COLS = [col for col in df_cleaned.columns if col not in [target_col]]
print(f"Number of features selected dynamically: {len(FEATURE_COLS)}")

# Impute features with median
for col in FEATURE_COLS:
    s = pd.to_numeric(df_cleaned[col], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    med = s.median()
    if pd.isna(med):
        med = 0.0
    df_cleaned[col] = s.fillna(med)

X = df_cleaned[FEATURE_COLS].values
y_label = df_cleaned[target_col].astype(str).str.strip().values

# Label Encoding
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)

print(f"Cleaned X shape: {X.shape}")
print(f"Classes: {encoder.classes_}")



## Stratified Train-Test Split & Scaling
Split data into stratified 80/20 train/test partitions and apply StandardScaler.

In [ ]:
# Stratified Train-Test Split and Standard Scaling (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train Shape: {X_train_scaled.shape} | Test Shape: {X_test_scaled.shape}")



## Base Model Instantiation
Configure optimized XGBoost, LightGBM, CatBoost, MLP, and Random Forest base models.

In [ ]:
# Model Configuration & Parameters
params_path = os.path.join(_project_root, "best_params.json")
best_params = {}
if os.path.isfile(params_path):
    try:
        with open(params_path, "r") as f:
            best_params = json.load(f)
        print("Loaded best_params.json successfully.")
    except Exception as e:
        print("Error loading best_params.json:", e)

# Compute global class weights to handle class imbalance across base models
from sklearn.utils.class_weight import compute_class_weight
unique_classes = np.unique(y_train)
global_weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)
global_weight_dict = dict(zip(unique_classes, global_weights))

xgb_params = best_params.get("xgboost", {
    "n_estimators": 468,
    "max_depth": 10,
    "learning_rate": 0.17549140891728818,
    "subsample": 0.9690394070981359,
    "colsample_bytree": 0.7725519831966194,
    "gamma": 1.0492767129301485e-08,
})
xgb_model = xgb.XGBClassifier(
    **xgb_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric="mlogloss",
    objective="multi:softprob",
    num_class=len(encoder.classes_),
    device="cuda" if CUDA_AVAILABLE else "cpu",
)

lgb_params = best_params.get("lightgbm", {
    "n_estimators": 499,
    "max_depth": 11,
    "num_leaves": 67,
    "learning_rate": 0.24625126683753454,
    "subsample": 0.6909971314141472,
    "colsample_bytree": 0.7554088091076056,
})
lgb_model = lgb.LGBMClassifier(
    **lgb_params,
    class_weight=global_weight_dict,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    objective="multiclass",
    num_class=len(encoder.classes_),
    device="gpu" if CUDA_AVAILABLE else "cpu",
)

cb_params = best_params.get("catboost", {
    "iterations": 500,
    "depth": 8,
    "learning_rate": 0.15,
    "l2_leaf_reg": 3.0,
})
cb_model = CatBoostClassifier(
    **cb_params,
    class_weights=global_weight_dict,
    loss_function="MultiClass",
    random_seed=RANDOM_STATE,
    thread_count=-1,
    verbose=False,
    task_type="GPU" if CUDA_AVAILABLE else "CPU",
)

# MLP Base Classifier (Neural representation)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_STATE
)

# Random Forest Base Classifier (Bagging Tree representation)
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=11,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


## Out-Of-Fold (OOF) prediction generation
Perform 10-Fold Stratified Cross-Validation to generate predictions for meta-classifier training.

In [ ]:
# 10-Fold Stratified Cross-Validation for OOF Predictions
print("\n--- Training Out-Of-Fold models for Stacking Meta-Features (10-Fold CV) ---")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
xgb_oof = np.zeros((len(X_train), len(encoder.classes_)))
lgb_oof = np.zeros((len(X_train), len(encoder.classes_)))
cb_oof = np.zeros((len(X_train), len(encoder.classes_)))
mlp_oof = np.zeros((len(X_train), len(encoder.classes_)))
rf_oof = np.zeros((len(X_train), len(encoder.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"  Processing Fold {fold + 1}...")
    X_tr_raw, y_tr = X_train[train_idx], y_train[train_idx]
    X_va_raw, y_va = X_train[val_idx], y_train[val_idx]

    # Fit scaler strictly on the training fold to avoid data leakage
    fold_scaler = StandardScaler()
    X_tr = fold_scaler.fit_transform(X_tr_raw)
    X_va = fold_scaler.transform(X_va_raw)

    # Compute class weights for this fold to handle class imbalance
    from sklearn.utils.class_weight import compute_class_weight
    unique_classes = np.unique(y_tr)
    computed_weights = compute_class_weight(
        class_weight='balanced', 
        classes=unique_classes, 
        y=y_tr
    )
    class_weight_dict = dict(zip(unique_classes, computed_weights))

    # Apply class weights dynamically to base classifiers
    lgb_fold_params = lgb_params.copy()
    lgb_fold_params['class_weight'] = class_weight_dict

    cb_fold_params = cb_params.copy()
    cb_fold_params['class_weights'] = class_weight_dict

    # Re-instantiate fold models
    xgb_m = xgb.XGBClassifier(**xgb_params, random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss", objective="multi:softprob", num_class=len(encoder.classes_), device="cuda" if CUDA_AVAILABLE else "cpu")
    lgb_m = lgb.LGBMClassifier(**lgb_fold_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, objective="multiclass", num_class=len(encoder.classes_), device="gpu" if CUDA_AVAILABLE else "cpu")
    cb_m = CatBoostClassifier(**cb_fold_params, loss_function="MultiClass", random_seed=RANDOM_STATE, thread_count=-1, verbose=False, task_type="GPU" if CUDA_AVAILABLE else "CPU")
    mlp_m = MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001, learning_rate_init=0.001, max_iter=300, early_stopping=True, validation_fraction=0.1, random_state=RANDOM_STATE)
    rf_m = RandomForestClassifier(n_estimators=300, max_depth=11, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)

    # Fit and predict probabilities
    xgb_m.fit(X_tr, y_tr)
    lgb_m.fit(X_tr, y_tr)
    cb_m.fit(X_tr, y_tr)
    mlp_m.fit(X_tr, y_tr)
    rf_m.fit(X_tr, y_tr)

    xgb_oof[val_idx] = xgb_m.predict_proba(X_va)
    lgb_oof[val_idx] = lgb_m.predict_proba(X_va)
    cb_oof[val_idx] = cb_m.predict_proba(X_va)
    mlp_oof[val_idx] = mlp_m.predict_proba(X_va)
    rf_oof[val_idx] = rf_m.predict_proba(X_va)

print("OOF predictions generated successfully.")


## Training Stacking Meta-Classifier
Train the Extra Trees meta-classifier on base model OOF predictions.

In [ ]:
# Stacking Meta-Classifier (Extra Trees with Tuned Depth)
print("\n--- Training Stacking Meta-Classifier (Extra Trees) ---")
oof_meta_features = np.hstack([xgb_oof, lgb_oof, cb_oof, mlp_oof, rf_oof])

meta_model = ExtraTreesClassifier(n_estimators=500, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)
meta_model.fit(oof_meta_features, y_train)
print("Stacking Meta-Classifier trained successfully on OOF features.")

# ----------------- Multi-Class Threshold Optimization -----------------
print("\n--- Generating OOF predictions for Meta-Classifier and Optimizing Thresholds ---")
from scipy.optimize import minimize

# Generate OOF probabilities for the meta-classifier itself using the same cross-validation splits
meta_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))
for fold, (train_idx, val_idx) in enumerate(skf.split(oof_meta_features, y_train)):
    X_tr_meta, y_tr_meta = oof_meta_features[train_idx], y_train[train_idx]
    X_va_meta, y_va_meta = oof_meta_features[val_idx], y_train[val_idx]
    
    meta_m = ExtraTreesClassifier(n_estimators=500, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1)
    meta_m.fit(X_tr_meta, y_tr_meta)
    meta_oof[val_idx] = meta_m.predict_proba(X_va_meta)

# Objective function: maximize weighted F1-score (minimize negative F1)
def objective(weights, probas, y_true):
    weighted_probas = probas * weights
    preds = np.argmax(weighted_probas, axis=1)
    return -f1_score(y_true, preds, average="weighted")

# Initial guess: all weights = 1.0
init_weights = np.ones(len(encoder.classes_))
# Bounds to keep weights positive and reasonable
bounds = [(0.1, 10.0)] * len(encoder.classes_)

print("Running Powell optimizer for decision thresholds...")
res = minimize(objective, init_weights, args=(meta_oof, y_train), method="Powell", bounds=bounds)
best_thresholds = res.x

print("\n=== Optimization Results ===")
for cls_name, w in zip(encoder.classes_, best_thresholds):
    print(f"  {cls_name}: {w:.4f}")


## Final Base Model Training & Testing
Fit base models on full training data and generate test set predictions.

In [ ]:
# Final Model Training on Full Training Set
print("\n--- Fitting final models on full scaled training set ---")
xgb_model.fit(X_train_scaled, y_train)
lgb_model.fit(X_train_scaled, y_train)
cb_model.fit(X_train_scaled, y_train)
mlp_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train_scaled, y_train)

# Generate test set base probabilities
xgb_proba = xgb_model.predict_proba(X_test_scaled)
lgb_proba = lgb_model.predict_proba(X_test_scaled)
cb_proba = cb_model.predict_proba(X_test_scaled)
mlp_proba = mlp_model.predict_proba(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)

# Concat test meta-features
test_meta_features = np.hstack([xgb_proba, lgb_proba, cb_proba, mlp_proba, rf_proba])

# Run meta-classifier predictions (Standard and Threshold-Optimized)
ensemble_proba = meta_model.predict_proba(test_meta_features)
ensemble_pred = np.argmax(ensemble_proba, axis=1)
ensemble_pred_opt = np.argmax(ensemble_proba * best_thresholds, axis=1)

# Base models predictions for comparison
xgb_pred = np.argmax(xgb_proba, axis=1)
lgb_pred = np.argmax(lgb_proba, axis=1)
cb_pred = np.argmax(cb_proba, axis=1)
mlp_pred = np.argmax(mlp_proba, axis=1)
rf_pred = np.argmax(rf_proba, axis=1)

print("Final predictions computed for standard and optimized stacks.")


## Performance Evaluation
Generate classification reports for all models.

In [ ]:
# Performance Evaluation
xgb_f1 = f1_score(y_test, xgb_pred, average="weighted")
xgb_kappa = cohen_kappa_score(y_test, xgb_pred)

lgb_f1 = f1_score(y_test, lgb_pred, average="weighted")
lgb_kappa = cohen_kappa_score(y_test, lgb_pred)

cb_f1 = f1_score(y_test, cb_pred, average="weighted")
cb_kappa = cohen_kappa_score(y_test, cb_pred)

mlp_f1 = f1_score(y_test, mlp_pred, average="weighted")
mlp_kappa = cohen_kappa_score(y_test, mlp_pred)

rf_f1 = f1_score(y_test, rf_pred, average="weighted")
rf_kappa = cohen_kappa_score(y_test, rf_pred)

ensemble_f1 = f1_score(y_test, ensemble_pred, average="weighted")
ensemble_kappa = cohen_kappa_score(y_test, ensemble_pred)
ensemble_report = classification_report(y_test, ensemble_pred, target_names=encoder.classes_, output_dict=True)

ensemble_opt_f1 = f1_score(y_test, ensemble_pred_opt, average="weighted")
ensemble_opt_kappa = cohen_kappa_score(y_test, ensemble_pred_opt)
ensemble_opt_report = classification_report(y_test, ensemble_pred_opt, target_names=encoder.classes_, output_dict=True)

print("\n=== XGBoost Test Metrics ===")
print(classification_report(y_test, xgb_pred, target_names=encoder.classes_))
print(f"Weighted F1: {xgb_f1:.4f}")

print("\n=== LightGBM Test Metrics ===")
print(classification_report(y_test, lgb_pred, target_names=encoder.classes_))
print(f"Weighted F1: {lgb_f1:.4f}")

print("\n=== CatBoost Test Metrics ===")
print(classification_report(y_test, cb_pred, target_names=encoder.classes_))
print(f"Weighted F1: {cb_f1:.4f}")

print("\n=== MLP Classifier Test Metrics ===")
print(classification_report(y_test, mlp_pred, target_names=encoder.classes_))
print(f"Weighted F1: {mlp_f1:.4f}")

print("\n=== Random Forest Test Metrics ===")
print(classification_report(y_test, rf_pred, target_names=encoder.classes_))
print(f"Weighted F1: {rf_f1:.4f}")

print("\n=== Standard Stacking Ensemble (XGB + LGB + CB + MLP + RF -> Extra Trees) ===")
print(classification_report(y_test, ensemble_pred, target_names=encoder.classes_))
print(f"Stacking Ensemble Weighted F1: {ensemble_f1:.4f}")
print(f"Stacking Ensemble Cohen Kappa: {ensemble_kappa:.4f}")

print("\n=== Threshold-Optimized Stacking Ensemble ===")
print(classification_report(y_test, ensemble_pred_opt, target_names=encoder.classes_))
print(f"Optimized Ensemble Weighted F1: {ensemble_opt_f1:.4f} (Change: {ensemble_opt_f1 - ensemble_f1:+.4f})")
print(f"Optimized Ensemble Cohen Kappa: {ensemble_opt_kappa:.4f}")


## Visualizations
Generate side-by-side confusion matrices, comparison bar charts, and per-class F1 plots.

In [ ]:
# Visualizing Confusion Matrices (7 Subplots Side-by-Side to include Optimized Stacking)
fig, axes = plt.subplots(1, 7, figsize=(32, 4.8))
predictions = [
    (xgb_pred, "XGBoost"),
    (lgb_pred, "LightGBM"),
    (cb_pred, "CatBoost"),
    (mlp_pred, "MLP Neural"),
    (rf_pred, "Random Forest"),
    (ensemble_pred, "Stacking (Std)"),
    (ensemble_pred_opt, "Stacking (Opt)")
]

for idx, (pred, title) in enumerate(predictions):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=encoder.classes_, yticklabels=encoder.classes_, ax=axes[idx], cbar=False)
    axes[idx].set_title(title)
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

plt.tight_layout()
figures_dir = os.path.join(_project_root, "figures")
os.makedirs(figures_dir, exist_ok=True)
plt.savefig(os.path.join(figures_dir, "stacking_confusion.png"), dpi=200, bbox_inches="tight")
plt.show()  # Display directly in notebook output

# Performance Comparison Chart including Optimized Stacking
models = ["XGBoost", "LightGBM", "CatBoost", "MLP Net", "Random Forest", "Stacking", "Stacking (Opt)"]
f1_scores = [xgb_f1, lgb_f1, cb_f1, mlp_f1, rf_f1, ensemble_f1, ensemble_opt_f1]
kappas = [xgb_kappa, lgb_kappa, cb_kappa, mlp_kappa, rf_kappa, ensemble_kappa, ensemble_opt_kappa]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(14, 5))
plt.bar(x - width / 2, f1_scores, width, label="Weighted F1", color="skyblue")
plt.bar(x + width / 2, kappas, width, label="Cohen Kappa", color="steelblue")
plt.xticks(x, models)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Comparison: Stacking Pipeline & Optimization")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "stacking_model_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()  # Display directly in notebook output

# Per-Class F1-Scores for Stacking (Std) vs Stacking (Opt)
class_f1_std = [ensemble_report[label]["f1-score"] for label in encoder.classes_]
class_f1_opt = [ensemble_opt_report[label]["f1-score"] for label in encoder.classes_]

x_cls = np.arange(len(encoder.classes_))
plt.figure(figsize=(10, 5))
plt.bar(x_cls - width/2, class_f1_std, width, label="Stacking (Standard)", color="lightgray")
plt.bar(x_cls + width/2, class_f1_opt, width, label="Stacking (Optimized)", color="steelblue")
plt.xticks(x_cls, list(encoder.classes_), rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("F1-score")
plt.xlabel("Emotion Class")
plt.title("Per-Class F1-Score: Standard vs. Threshold-Optimized Stacking")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "stacking_class_f1.png"), dpi=200, bbox_inches="tight")
plt.show()  # Display directly in notebook output


## Serialization
Serialize the trained models, preprocessing tools, and reports to disk.

In [ ]:
# Save Trained Models, Preprocessors, and Optimized Thresholds
print("\n--- Serializing production artifacts for live backend ---")
models_dir = os.path.join(_project_root, 'models')
os.makedirs(models_dir, exist_ok=True)

saved_files = []
def save_asset(obj, name):
    path = os.path.join(models_dir, name)
    joblib.dump(obj, path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    saved_files.append(f"  - {name} ({size_mb:.2f} MB)")

save_asset(xgb_model, 'ser_xgb_model.joblib')
save_asset(lgb_model, 'ser_lgb_model.joblib')
save_asset(cb_model,  'ser_cb_model.joblib')
save_asset(mlp_model, 'ser_mlp_model.joblib')
save_asset(rf_model,  'ser_rf_model.joblib')
save_asset(meta_model, 'ser_meta_model.joblib')
save_asset(scaler,    'ser_ensemble_scaler.joblib')
save_asset(encoder,   'ser_ensemble_encoder.joblib')
save_asset(best_thresholds, 'ser_ensemble_thresholds.joblib')

print("SUCCESS: All base models, preprocessors, meta-classifier, and thresholds saved to disk:")
for f in saved_files:
    print(f)

# Save Stacking Report
report_path = os.path.join(_project_root, 'stacking_report.txt')
with open(report_path, 'w') as f:
    f.write('=== Speech Emotion Recognition Ultimate Stacking Ensemble Training Report ===\n\n')
    f.write('=== XGBoost Weighted F1 ===\n')
    f.write(f'{xgb_f1:.4f}\n\n')
    f.write('=== LightGBM Weighted F1 ===\n')
    f.write(f'{lgb_f1:.4f}\n\n')
    f.write('=== CatBoost Weighted F1 ===\n')
    f.write(f'{cb_f1:.4f}\n\n')
    f.write('=== MLP Base Weighted F1 ===\n')
    f.write(f'{mlp_f1:.4f}\n\n')
    f.write('=== Random Forest Weighted F1 ===\n')
    f.write(f'{rf_f1:.4f}\n\n')
    f.write('=== Stacking Ensemble (XGB + LGB + CB + MLP + RF -> Extra Trees) ===\n')
    f.write(classification_report(y_test, ensemble_pred, target_names=encoder.classes_))
    f.write(f'\nEnsemble Weighted F1: {ensemble_f1:.4f}\n')
    f.write(f'Ensemble Cohen Kappa: {ensemble_kappa:.4f}\n\n')
    f.write('=== Threshold-Optimized Stacking Ensemble ===\n')
    f.write(classification_report(y_test, ensemble_pred_opt, target_names=encoder.classes_))
    f.write(f'\nOptimized Ensemble Weighted F1: {ensemble_opt_f1:.4f}\n')
    f.write(f'Optimized Ensemble Cohen Kappa: {ensemble_opt_kappa:.4f}\n')
    f.write(f'Optimized Threshold Multipliers: {list(best_thresholds)}\n')

print(f'Saved performance report to {report_path}')
